# Skills Based Agents

Give an OpenAI managed agent a reusable skill, run a sales-report task, and verify both the skill execution and its output files.

[Open in Google Colab](https://colab.research.google.com/github/BrightPool/udemy-prompt-engineering-course/blob/main/building_ai_agents/skills_based_agents.ipynb) · [Official Skills guide](https://developers.openai.com/api/docs/guides/tools-skills)

**Part 10: Building AI Agents - Coding.** This lesson builds on the Managed Agents API notebook in Part 7.

A skill packages **how to do a task** in a folder containing `SKILL.md` and optional scripts, references, or templates. The harness discovers the skill's name and description; the model can read its full instructions when the task needs them. A tool supplies an action, while a skill describes the process for using actions well.

**Filming path (about 4-5 minutes):** pre-run setup, inspect the skill manifest, show how the folder is attached, run one short task, and check the downloaded report and execution trace. The supporting script and stream helper are supplied.

This example uses the **Agents API** (`client.beta.agents`), not the local Agents SDK's `Agent`/`Runner` interface. You need Agents API access, API billing, and an application key with `api.agents.read`, `api.agents.write`, and `api.responses.write`. Hosted environments and model usage are billed separately. We use synthetic data and disable the sandbox's network access.

In [ ]:
%pip install openai --upgrade --quiet

In [ ]:
RUN_DEMO = True
CLEANUP = True

In [ ]:
import os
import json
import csv
import base64
import time
import subprocess
from pathlib import Path
from getpass import getpass
from openai import OpenAI
from IPython.display import display, Markdown, FileLink

if not isinstance(RUN_DEMO, bool):
    raise TypeError("RUN_DEMO must be True or False, not text.")
if RUN_DEMO and not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
client = OpenAI(timeout=180, max_retries=1) if RUN_DEMO else None
MODEL = os.getenv("OPENAI_AGENTS_MODEL", "gpt-6-astra")
print("Model:", MODEL)

## 1. Create the skill manifest

The front matter gives the skill its **name** and **description**. The description says both what the skill does and when it should be used. The body contains the repeatable workflow and points to its supporting script.

We use a clear report convention: calculate from the CSV, produce three named files, and check the result. The short task prompt later will not repeat these instructions.

In [ ]:
workdir = Path("skills_based_agents_demo")
skill_dir = workdir / "skills" / "sales-report"
(skill_dir / "scripts").mkdir(parents=True, exist_ok=True)
SKILL_MD = """---
name: sales-report
description: Analyse a sales CSV and create a checked revenue report with regional totals. Use when a user asks for a sales summary or a repeatable revenue report.
---

# Sales report workflow

1. Read the input CSV. It must contain region and revenue columns.
2. Run the supporting script from this skill directory:
   `python scripts/summarize_sales.py /workspace/sales.csv /workspace/outputs`
   Resolve the script path relative to this SKILL.md; do not assume the working directory is the skill folder.
3. The script writes report.md, regional_totals.csv and summary.json.
4. Read the generated files and check that the regional totals add up to the grand total.
5. Keep the report's heading and table format. Do not replace the script with invented calculations.
6. Return the actual grand total, top region and output filenames. Do not claim a step succeeded unless you checked its result.

This task uses synthetic data. It needs no network access, credentials or additional packages.
"""
(skill_dir / "SKILL.md").write_text(SKILL_MD, encoding="utf-8")
print(SKILL_MD)

## 2. Add a supporting script and a tiny input file

A skill can include code the agent can run. Here the script performs the deterministic arithmetic and writes a standard report, CSV, and JSON summary. It needs only Python's standard library. The manifest refers to this script by a path relative to the skill folder.

We test the helper locally before making it available to an agent. This separates a broken script from a problem with the agent's use of the skill.

In [ ]:
SCRIPT = r'''import csv
import json
import sys
from decimal import Decimal
from pathlib import Path

source = Path(sys.argv[1])
destination = Path(sys.argv[2])
destination.mkdir(parents=True, exist_ok=True)
totals = {}
with source.open(newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    if not {"region", "revenue"}.issubset(reader.fieldnames or []):
        raise ValueError("Input CSV needs region and revenue columns.")
    for row in reader:
        region = row["region"].strip()
        amount = Decimal(row["revenue"])
        if not region or not amount.is_finite():
            raise ValueError("Each row needs a region and a finite revenue value.")
        totals[region] = totals.get(region, Decimal("0")) + amount
if not totals:
    raise ValueError("Input CSV has no data rows.")
grand_total = sum(totals.values(), Decimal("0"))
top_region = max(totals, key=totals.get)
with (destination / "regional_totals.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["region", "total_revenue"])
    writer.writerows((region, f"{amount:.2f}") for region, amount in sorted(totals.items()))
summary = {"skill": "sales-report", "grand_total": float(grand_total),
           "top_region": top_region, "regions": len(totals)}
(destination / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
lines = ["# Sales report", "", f"Grand total: {grand_total:.2f}",
         f"Top region: {top_region} ({totals[top_region]:.2f})", "",
         "| Region | Revenue |", "|---|---:|"]
lines += [f"| {region} | {amount:.2f} |" for region, amount in sorted(totals.items())]
(destination / "report.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(json.dumps(summary))
'''
script_path = skill_dir / "scripts" / "summarize_sales.py"
script_path.write_text(SCRIPT, encoding="utf-8")
sales_path = workdir / "sales.csv"
sales_path.write_text(
    "month,region,revenue\nJan,North,120\nJan,South,100\n"
    "Feb,North,140\nFeb,South,120\nMar,North,160\nMar,South,140\n",
    encoding="utf-8",
)
local_outputs = workdir / "local_check"
import sys
subprocess.run([sys.executable, str(script_path), str(sales_path), str(local_outputs)], check=True)
expected_summary = {"skill": "sales-report", "grand_total": 780.0, "top_region": "North", "regions": 2}
assert json.loads((local_outputs / "summary.json").read_text()) == expected_summary
display(Markdown((local_outputs / "report.md").read_text()))

## 3. Attach the skill folder to the sandbox

For the **Agents API**, put the skill files in the environment and register the **parent directory** in `capability_directories`. Our folder will be `/workspace/skills/sales-report`, so the registered parent is `/workspace/skills`.

We send these small files inline as base64. Environment files are installed before setup commands run, so the skill folder exists before the harness discovers it. The setup check fails early if the manifest or supporting script is missing.

The Responses API has a separate hosted-shell skill attachment format (`tools[].environment.skills` with `skill_reference`). This notebook uses the Agents API directory-discovery path.

In [ ]:
def inline_file(local_path, remote_path):
    return {"type": "inline", "path": remote_path,
            "data": base64.b64encode(local_path.read_bytes()).decode("ascii")}

environment = {
    "type": "openai_hosted",
    "network": {"access": "disabled"},
    "files": [
        inline_file(skill_dir / "SKILL.md", "/workspace/skills/sales-report/SKILL.md"),
        inline_file(script_path, "/workspace/skills/sales-report/scripts/summarize_sales.py"),
        inline_file(sales_path, "/workspace/sales.csv"),
    ],
    "setup_commands": [{"command": (
        "test -f /workspace/skills/sales-report/SKILL.md && "
        "test -f /workspace/skills/sales-report/scripts/summarize_sales.py")}],
    "capability_directories": ["/workspace/skills"],
}
print("Skill discovery directory:", environment["capability_directories"])
print("Files available to the agent:", [f["path"] for f in environment["files"]])

## 4. Run the agent with a short task prompt

The agent's general instructions stay short. The skill holds the detailed workflow. We explicitly request `sales-report` for this first lesson; in a larger application, the model can select relevant discovered skills using their descriptions.

The helper prints progress and waits for the root turn to finish. A completed turn is followed by output checks, not treated as proof that every instruction was followed.

In [ ]:
session_id = None

def consume_turn(events):
    global session_id
    deadline = time.monotonic() + 180
    for event in events:
        if time.monotonic() > deadline:
            raise TimeoutError("Demo time limit reached. Inspect the session before retrying.")
        data = event.model_dump()
        kind = data["type"]
        if kind == "agent.session.created":
            session_id = data["session"]["id"]
            print("Session:", session_id)
        else:
            session_id = data.get("session_id", session_id)
        if kind == "agent.session.turn.output_text.delta":
            print(data["delta"], end="", flush=True)
        if kind in {"error", "agent.session.failed", "agent.session.environment.failed"}:
            raise RuntimeError(f"Agent lifecycle failure: {data}")
        if kind == "agent.session.requires_action":
            raise RuntimeError("Unexpected required action. Inspect session.required_actions.")
        if (data.get("turn") or {}).get("subagent_id") is None:
            if kind in {"agent.session.turn.failed", "agent.session.turn.cancelled"}:
                raise RuntimeError(f"Turn did not complete: {data}")
            if kind == "agent.session.turn.completed":
                print("\nTurn completed:", data["turn_id"])
                return data["turn_id"]
    raise RuntimeError("Stream closed before completion. Retrieve saved session items.")

In [ ]:
turn_id = None
if RUN_DEMO:
    try:
        with client.beta.agents.sessions.create(
            agent={"model": MODEL, "instructions": (
                "Use relevant skills for the task. Read their instructions before acting. "
                "Report actual results briefly. Do not delegate work or use network access. "
                "Use plain text without em dashes.")},
            environment=environment,
            input=("Use the sales-report skill to analyse /workspace/sales.csv. "
                   "First print its SKILL.md in a shell command so we can inspect it in the execution trace. "
                   "Follow its instructions and save its outputs."),
            stream=True,
        ) as events:
            turn_id = consume_turn(events)
    except Exception:
        if session_id:
            client.beta.agents.sessions.delete(session_id)
            session_id = None
        raise

## 5. Check that the agent read and executed the skill

Attaching a skill is not proof that it was used. Inspect saved command-execution items for a successful read of `SKILL.md` and a successful run of the supplied helper. Then check the files separately.

We explicitly requested a visible manifest read for this demonstration. Check its name in the command output, rather than accepting a model-written claim such as "I used the skill." This confirms the requested read and script execution; it is not a general test of every harness's internal skill-loading mechanism. A production evaluation should also check its own task-specific requirements.

In [ ]:
if RUN_DEMO:
    commands = [item for item in client.beta.agents.sessions.items.list(session_id, order="asc")
                if item.type == "command_execution" and item.turn_id == turn_id]
    for item in commands:
        print("Command:", item.command, "Exit code:", item.exit_code)
    read_skill = any("SKILL.md" in item.command and "name: sales-report" in (item.output or "")
                     and item.exit_code == 0 for item in commands)
    ran_helper = any("summarize_sales.py" in item.command and item.exit_code == 0
                     and '"grand_total": 780.0' in (item.output or "") for item in commands)
    assert read_skill, "No successful manifest read in the trace. Inspect the commands before claiming skill use."
    assert ran_helper, "No successful supporting-script execution in the trace. Inspect the commands."
    print("Verified: the agent read the skill manifest and ran its supporting script.")

## 6. Download and verify the report artifacts

The report format and totals come from the skill's supporting script. Check both: correct numbers alone do not tell us whether the agent followed the requested process.

In [ ]:
outputs = workdir / "agent_outputs"
outputs.mkdir(exist_ok=True)
if RUN_DEMO:
    artifacts = list(client.beta.agents.sessions.artifacts.list(session_id, order="desc"))
    for filename in ("report.md", "regional_totals.csv", "summary.json"):
        path = "/workspace/outputs/" + filename
        artifact = next((a for a in artifacts if a.path == path and a.turn_id == turn_id), None)
        if artifact is None:
            raise FileNotFoundError(f"Missing skill output: {path}")
        destination = outputs / filename
        with client.beta.agents.sessions.artifacts.with_streaming_response.content(
            artifact.id, session_id=session_id,
        ) as content:
            content.stream_to_file(destination)
        display(FileLink(str(destination)))
    summary = json.loads((outputs / "summary.json").read_text())
    assert summary == expected_summary, f"Unexpected summary: {summary}"
    with (outputs / "regional_totals.csv").open() as f:
        totals = {row["region"]: float(row["total_revenue"]) for row in csv.DictReader(f)}
    assert totals == {"North": 420, "South": 360}
    report = (outputs / "report.md").read_text()
    assert report == (local_outputs / "report.md").read_text(), "The standard report format changed."
    display(Markdown(report))
    print("Verified: the skill's output schema, report format and calculated totals.")

## 7. Clean up the remote session

The generated skill folder and downloaded results remain in `skills_based_agents_demo/`. Save them to your own computer or Drive if you want to retain them beyond the Colab session.

In [ ]:
if RUN_DEMO and CLEANUP and session_id:
    client.beta.agents.sessions.delete(session_id)
    session_id = None
    print("Deleted this demo's remote session; local skill files and outputs remain.")

## Try it yourself

Change the revenue values and update the expected checks, then rerun. For a style exercise, edit the supporting script's report heading, rerun the local check, and ask the agent to use the updated skill. The task prompt can stay the same.

## Summary

- A skill is a reusable folder with `SKILL.md` and optional supporting files.
- Name and description support discovery; the full instructions guide execution when needed.
- Agents API skills live in the sandbox, with their parent registered in `capability_directories`.
- Run a task with the skill available, then check evidence of skill use and the resulting artifacts.
- Review the instructions and code you attach. Skills do not replace tool permissions or output evaluation.

**Troubleshooting:** update the SDK if `beta.agents` is missing; check project access and key permissions for 403 errors; capability directories must be absolute paths inside the sandbox and exist before discovery. If a later check fails, inspect the saved commands and artifacts before running cleanup. Only these three small synthetic files are uploaded; the API key stays in the notebook client.

Sources: [Skills](https://developers.openai.com/api/docs/guides/tools-skills#agents-api), [Hosted sandbox setup](https://developers.openai.com/api/docs/guides/agents-api/environments/openai-hosted), [Session events](https://developers.openai.com/api/docs/guides/agents-api/sessions/events), [Published artifacts](https://developers.openai.com/api/docs/guides/agents-api/environments/files), [Agent Skills specification](https://agentskills.io/specification).